In [205]:
import pandas as pd
import numpy as np

# 1. Load the Core Files
results = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
teams = pd.read_csv('data/MTeams.csv')
seeds = pd.read_csv('data/MNCAATourneySeeds.csv')

# 2. Clean Seeds: Extract numeric value (e.g., 'W01' -> 1)
seeds['Seed'] = seeds['Seed'].apply(lambda x: int(''.join(filter(str.isdigit, x))))

# 3. Separate Winning and Losing stats to capture Opponent Scoring
# For Winner: Opponent Score is the points allowed (LScore)
w_stats = results[['Season', 'WTeamID', 'WScore', 'WFGM', 'WFGA', 'WFGM3', 'WAst', 'LScore']].copy()
w_stats.columns = ['Season', 'TeamID', 'Score', 'FGM', 'FGA', 'FGM3', 'Ast', 'OppScore']

# For Loser: Opponent Score is the points allowed (WScore)
l_stats = results[['Season', 'LTeamID', 'LScore', 'LFGM', 'LFGA', 'LFGM3', 'LAst', 'WScore']].copy()
l_stats.columns = ['Season', 'TeamID', 'Score', 'FGM', 'FGA', 'FGM3', 'Ast', 'OppScore']

# 4. Aggregate and calculate season averages
team_stats = pd.concat([w_stats, l_stats]).groupby(['Season', 'TeamID']).mean()

# 5. Calculate Efficiency and Performance Metrics
# eFG% = Effective Field Goal Percentage
team_stats['eFG'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA']
# PPS = Points Per Shot
team_stats['PPS'] = team_stats['Score'] / team_stats['FGA']

# 6. Extract the most recent season (2026) and flatten the index for retrieval
latest_season = team_stats.index.get_level_values(0).max()
latest_stats = team_stats.xs(latest_season, level=0).reset_index()

print(f"🏀 Season {latest_season} Data Loaded! All metrics are ready for March Madness.")

🏀 Season 2026 Data Loaded! All metrics are ready for March Madness.


In [206]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Prepare Training Data
train_results = results.merge(seeds, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'WSeed'}).drop('TeamID', axis=1)
train_results = train_results.merge(seeds, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'LSeed'}).drop('TeamID', axis=1)

# 2. Build the Features
side_a = pd.DataFrame()
side_a['ScoreDiff'] = train_results['WScore'] - train_results['LScore']
side_a['FGM_Diff'] = train_results['WFGM'] - train_results['LFGM']
side_a['Ast_Diff'] = train_results['WAst'] - train_results['LAst']
side_a['SeedDiff'] = train_results['WSeed'] - train_results['LSeed']
side_a['Result'] = 1

side_b = pd.DataFrame()
side_b['ScoreDiff'] = train_results['LScore'] - train_results['WScore']
side_b['FGM_Diff'] = train_results['LFGM'] - train_results['WFGM']
side_b['Ast_Diff'] = train_results['LAst'] - train_results['WAst']
side_b['SeedDiff'] = train_results['LSeed'] - train_results['WSeed']
side_b['Result'] = 0

train_df = pd.concat([side_a, side_b]).dropna()

# 3. Final Training (Using .values to strip names and prevent warnings)
scaler = StandardScaler()
features = ['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff']

# THE KEY FIX: We use .values here so the model doesn't expect names later
X = scaler.fit_transform(train_df[features].values)
y = train_df['Result'].values

model = LogisticRegression(C=0.001)
model.fit(X, y)

print("🧠 Model Trained & Locked! Raw arrays synced for maximum precision.")

🧠 Model Trained & Locked! Raw arrays synced for maximum precision.


In [207]:
def predict_game_v5(team1_name, team2_name, seed1, seed2):
    try:
        # Match names to IDs
        t1_id = teams[teams['TeamName'] == team1_name]['TeamID'].values[0]
        t2_id = teams[teams['TeamName'] == team2_name]['TeamID'].values[0]

        # Get stats
        stats1 = latest_stats[latest_stats['TeamID'] == t1_id].iloc[0]
        stats2 = latest_stats[latest_stats['TeamID'] == t2_id].iloc[0]

        # Features for the ML model
        input_data = np.array([[
            stats1['Score'] - stats2['Score'],
            stats1['FGM'] - stats2['FGM'],
            stats1['Ast'] - stats2['Ast'],
            seed1 - seed2
        ]])

        X_scaled = scaler.transform(input_data)
        stat_prob = model.predict_proba(X_scaled)[0][1]

        # --- THE POWER BLEND (Adjusted for Seed Weight) ---
        seed_diff = seed2 - seed1
        # Increased to 0.07 to place higher emphasis on historical seed performance
        seed_only_prob = 0.5 + (seed_diff * 0.07)

        final_prob = (stat_prob * 0.50) + (seed_only_prob * 0.50)

        # --- TOURNAMENT OVERRIDES ---

        # 1. Elite Seed Protection: Adjustments for Top-Tier vs Lower Seeds
        if seed1 <= 2 and seed2 >= 10: final_prob += 0.08
        if seed2 <= 2 and seed1 >= 10: final_prob -= 0.08

        # 2. Defensive Efficiency Bonus: Reward teams allowing < 65 PPG
        if stats1['OppScore'] < 65: final_prob += 0.04
        if stats2['OppScore'] < 65: final_prob -= 0.04

        # 3. Tournament Pedigree: Historical Performance Adjustment
        pedigree_teams = ['Michigan St', 'Duke', 'Kansas', 'Connecticut', 'Kentucky']
        if team1_name in pedigree_teams: final_prob += 0.03
        if team2_name in pedigree_teams: final_prob -= 0.03

        return max(0.05, min(0.95, final_prob))

    except Exception as e:
        # Display specific error for troubleshooting
        print(f"Error for {team1_name} vs {team2_name}: {e}")
        return 0.5001

In [34]:
# =================================================================
# 2026 ROUND OF 64: PREDICTIONS & RESULTS
# =================================================================

# Data Format: (Team1, Team2, Seed1, Seed2, Original_Prediction, Original_Prob, Actual_Winner)
round_64_master = [
    ("Duke", "Siena", 1, 16, "Duke", 0.950, "Duke"),
    ("Ohio St", "TCU", 8, 9, "Ohio St", 0.555, "TCU"),
    ("St John's", "Northern Iowa", 5, 12, "St John's", 0.906, "St John's"),
    ("Kansas", "Cal Baptist", 4, 13, "Kansas", 0.950, "Kansas"),
    ("Louisville", "South Florida", 6, 11, "Louisville", 0.700, "Louisville"),
    ("Michigan St", "N Dakota St", 3, 14, "Michigan St", 0.950, "Michigan St"),
    ("UCLA", "UCF", 7, 10, "UCLA", 0.548, "UCLA"),
    ("Connecticut", "Furman", 2, 15, "Connecticut", 0.950, "Connecticut"),
    ("Arizona", "LIU Brooklyn", 1, 16, "Arizona", 0.950, "Arizona"),
    ("Villanova", "Utah St", 8, 9, "Utah St", 0.551, "Utah St"),
    ("Wisconsin", "High Point", 5, 12, "Wisconsin", 0.739, "High Point"),
    ("Arkansas", "Hawaii", 4, 13, "Arkansas", 0.950, "Arkansas"),
    ("BYU", "Texas", 6, 11, "BYU", 0.767, "Texas"),
    ("Gonzaga", "Kennesaw", 3, 14, "Gonzaga", 0.950, "Gonzaga"),
    ("Miami FL", "Missouri", 7, 10, "Miami FL", 0.688, "Miami FL"),
    ("Purdue", "Queens NC", 2, 15, "Purdue", 0.950, "Purdue"),
    ("Michigan", "Howard", 1, 16, "Michigan", 0.950, "Michigan"),
    ("Georgia", "St Louis", 8, 9, "Georgia", 0.539, "St Louis"),
    ("Texas Tech", "Akron", 5, 12, "Texas Tech", 0.684, "Texas Tech"),
    ("Alabama", "Hofstra", 4, 13, "Alabama", 0.950, "Alabama"),
    ("Tennessee", "Miami OH", 6, 11, "Tennessee", 0.622, "Tennessee"),
    ("Virginia", "Wright St", 3, 14, "Virginia", 0.950, "Virginia"),
    ("Kentucky", "Santa Clara", 7, 10, "Kentucky", 0.601, "Kentucky"),
    ("Iowa St", "Tennessee St", 2, 15, "Iowa St", 0.950, "Iowa St"),
    ("Florida", "Prairie View", 1, 16, "Florida", 0.950, "Florida"),
    ("Clemson", "Iowa", 8, 9, "Clemson", 0.507, "Iowa"),
    ("Vanderbilt", "McNeese St", 5, 12, "Vanderbilt", 0.929, "Vanderbilt"),
    ("Nebraska", "Troy", 4, 13, "Nebraska", 0.913, "Nebraska"),
    ("North Carolina", "VCU", 6, 11, "North Carolina", 0.702, "VCU"),
    ("Illinois", "Penn", 3, 14, "Illinois", 0.950, "Illinois"),
    ("St Mary's CA", "Texas A&M", 7, 10, "St Mary's CA", 0.503, "Texas A&M"),
    ("Houston", "Idaho", 2, 15, "Houston", 0.950, "Houston")
]

print(f"{'2026 ROUND OF 64 MATCHUP':<40} | {'PREDICTED (CONFIDENCE)':<25} | {'RESULT':<15} | STATUS")
print("-" * 105)

correct_picks = 0
total_games = len(round_64_master)

for t1, t2, s1, s2, pred_win, conf, actual in round_64_master:
    is_correct = (pred_win == actual)
    if is_correct: correct_picks += 1
    status = "✅" if is_correct else "❌"

    matchup = f"{t1} ({s1}) vs {t2} ({s2})"
    pred_str = f"{pred_win} ({conf:.1%})"

    print(f"{matchup:<40} | {pred_str:<25} | {actual:<15} | {status}")

accuracy_pct = (correct_picks / total_games) * 100
print("-" * 105)
print(f"ROUND PERFORMANCE: {correct_picks}/{total_games} Correct ({accuracy_pct:.2f}%)")

2026 ROUND OF 64 MATCHUP                 | PREDICTED (CONFIDENCE)    | RESULT          | STATUS
---------------------------------------------------------------------------------------------------------
Duke (1) vs Siena (16)                   | Duke (95.0%)              | Duke            | ✅
Ohio St (8) vs TCU (9)                   | Ohio St (55.5%)           | TCU             | ❌
St John's (5) vs Northern Iowa (12)      | St John's (90.6%)         | St John's       | ✅
Kansas (4) vs Cal Baptist (13)           | Kansas (95.0%)            | Kansas          | ✅
Louisville (6) vs South Florida (11)     | Louisville (70.0%)        | Louisville      | ✅
Michigan St (3) vs N Dakota St (14)      | Michigan St (95.0%)       | Michigan St     | ✅
UCLA (7) vs UCF (10)                     | UCLA (54.8%)              | UCLA            | ✅
Connecticut (2) vs Furman (15)           | Connecticut (95.0%)       | Connecticut     | ✅
Arizona (1) vs LIU Brooklyn (16)         | Arizona (95.0%)           |

In [31]:
# =================================================================
# 2026 ROUND OF 32: PREDICTIONS & RESULTS
# =================================================================

# Data Format: (Team1, Team2, Seed1, Seed2, Predicted_Winner, Confidence, Actual_Winner)
round_32_complete = [
    # --- East Region ---
    ("Duke", "TCU", 1, 9, "Duke", 0.950, "Duke"),
    ("St John's", "Kansas", 5, 4, "St John's", 0.528, "St John's"),
    ("Louisville", "Michigan St", 6, 3, "Michigan St", 0.554, "Michigan St"),
    ("UCLA", "Connecticut", 7, 2, "Connecticut", 0.781, "Connecticut"),

    # --- West Region ---
    ("Arizona", "Utah St", 1, 9, "Arizona", 0.907, "Arizona"),
    ("Vanderbilt", "Nebraska", 5, 4, "Vanderbilt", 0.564, "Nebraska"),
    ("Miami FL", "Purdue", 7, 2, "Purdue", 0.727, "Purdue"),
    ("Texas", "Gonzaga", 11, 3, "Gonzaga", 0.937, "Texas"),

    # --- Midwest Region ---
    ("Michigan", "St Louis", 1, 9, "Michigan", 0.856, "Michigan"),
    ("Texas Tech", "Alabama", 5, 4, "Alabama", 0.672, "Alabama"),
    ("Tennessee", "Virginia", 6, 3, "Virginia", 0.640, "Tennessee"),
    ("Kentucky", "Iowa St", 7, 2, "Iowa St", 0.721, "Iowa St"),

    # --- South Region ---
    ("Florida", "Iowa", 1, 9, "Florida", 0.950, "Iowa"),
    ("Arkansas", "High Point", 4, 12, "Arkansas", 0.904, "Arkansas"),
    ("VCU", "Illinois", 11, 3, "Illinois", 0.868, "Illinois"),
    ("Texas A&M", "Houston", 10, 2, "Houston", 0.790, "Houston")
]

print(f"{'2026 ROUND OF 32 MATCHUP':<36} | {'PREDICTED (CONFIDENCE)':<25} | {'RESULT':<12} | STATUS")
print("-" * 105)

correct_picks = 0
total_games = len(round_32_complete)

for t1, t2, s1, s2, pred_win, conf, actual in round_32_complete:
    is_correct = (pred_win == actual)
    if is_correct: correct_picks += 1
    status = "✅" if is_correct else "❌"

    matchup = f"{t1} ({s1}) vs {t2} ({s2})"
    pred_str = f"{pred_win} ({conf:.1%})"

    print(f"{matchup:<36} | {pred_str:<25} | {actual:<12} | {status}")

# Calculate final stats
accuracy_pct = (correct_picks / total_games) * 100

print("-" * 105)
print(f"ROUND PERFORMANCE: {correct_picks}/{total_games} Correct ({accuracy_pct:.1f}%)")

2026 ROUND OF 32 MATCHUP             | PREDICTED (CONFIDENCE)    | RESULT       | STATUS
---------------------------------------------------------------------------------------------------------
Duke (1) vs TCU (9)                  | Duke (95.0%)              | Duke         | ✅
St John's (5) vs Kansas (4)          | St John's (52.8%)         | St John's    | ✅
Louisville (6) vs Michigan St (3)    | Michigan St (55.4%)       | Michigan St  | ✅
UCLA (7) vs Connecticut (2)          | Connecticut (78.1%)       | Connecticut  | ✅
Arizona (1) vs Utah St (9)           | Arizona (90.7%)           | Arizona      | ✅
Vanderbilt (5) vs Nebraska (4)       | Vanderbilt (56.4%)        | Nebraska     | ❌
Miami FL (7) vs Purdue (2)           | Purdue (72.7%)            | Purdue       | ✅
Texas (11) vs Gonzaga (3)            | Gonzaga (93.7%)           | Texas        | ❌
Michigan (1) vs St Louis (9)         | Michigan (85.6%)          | Michigan     | ✅
Texas Tech (5) vs Alabama (4)        | Alabama (6

In [36]:
# =================================================================
# 2026 SWEET 16: PREDICTIONS & RESULTS
# =================================================================

# Data Format: (Team1, Team2, Seed1, Seed2, Predicted_Winner, Confidence, Actual_Winner)
# Note: Actual_Winner is left as "" until games are played.
sweet_16_matchups = [
    # East Region
    ("Duke", "St John's", 1, 5, "Duke", 0.742, ""),
    ("Michigan St", "Connecticut", 3, 2, "Connecticut", 0.615, ""),

    # West Region
    ("Arizona", "Arkansas", 1, 4, "Arizona", 0.718, ""),
    ("Purdue", "Texas", 2, 11, "Purdue", 0.824, ""),

    # Midwest Region
    ("Michigan", "Alabama", 1, 4, "Michigan", 0.685, ""),
    ("Tennessee", "Iowa St", 6, 2, "Iowa St", 0.592, ""),

    # South Region
    ("Iowa", "Nebraska", 9, 4, "Nebraska", 0.541, ""),
    ("Illinois", "Houston", 3, 2, "Houston", 0.655, "")
]

print(f"{'2026 SWEET 16 MATCHUP':<36} | {'PREDICTED (CONFIDENCE)':<25} | {'RESULT':<12} | STATUS")
print("-" * 105)

correct_picks = 0
total_games = 0

for t1, t2, s1, s2, pred_win, conf, actual in sweet_16_matchups:
    total_games += 1

    # Status Logic: Only show emoji if a result has been entered
    status = ""
    if actual != "":
        if pred_win == actual:
            correct_picks += 1
            status = "✅"
        else:
            status = "❌"

    matchup_label = f"{t1} ({s1}) vs {t2} ({s2})"
    pred_label = f"{pred_win} ({conf:.1%})"

    print(f"{matchup_label:<36} | {pred_label:<25} | {actual:<12} | {status}")

# Final summary section
print("-" * 105)
if any(game[6] != "" for game in sweet_16_matchups):
    accuracy_pct = (correct_picks / total_games) * 100
    print(f"ROUND PERFORMANCE: {correct_picks}/{total_games} Correct ({accuracy_pct:.1f}%)")
else:
    print(f"ROUND PERFORMANCE: 0/{total_games} Correct (Games Pending)")

2026 SWEET 16 MATCHUP                | PREDICTED (CONFIDENCE)    | RESULT       | STATUS
---------------------------------------------------------------------------------------------------------
Duke (1) vs St John's (5)            | Duke (74.2%)              |              | 
Michigan St (3) vs Connecticut (2)   | Connecticut (61.5%)       |              | 
Arizona (1) vs Arkansas (4)          | Arizona (71.8%)           |              | 
Purdue (2) vs Texas (11)             | Purdue (82.4%)            |              | 
Michigan (1) vs Alabama (4)          | Michigan (68.5%)          |              | 
Tennessee (6) vs Iowa St (2)         | Iowa St (59.2%)           |              | 
Iowa (9) vs Nebraska (4)             | Nebraska (54.1%)          |              | 
Illinois (3) vs Houston (2)          | Houston (65.5%)           |              | 
---------------------------------------------------------------------------------------------------------
ROUND PERFORMANCE: 0/8 Correct (Gam